# BigQuery サンプルデータ分析

このノートブックでは、BigQuery の公開データセットを使ってデータ分析の基礎を学びます。

## 学べること
1. **SQL** でデータを取得する方法
2. **pandas** でデータを加工する方法
3. **グラフ** でデータを可視化する方法
4. **統計分析** の基本（平均、相関など）

## 使用するデータ
Google が公開している **bigquery-public-data** から以下のデータを使います：
- `austin_bikeshare`: テキサス州オースティンの自転車シェアリングデータ

---
## 0. 準備：ライブラリの読み込み

In [ ]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 日本語フォント設定（Windows）
plt.rcParams['font.family'] = 'MS Gothic'
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

# BigQuery クライアントを作成
client = bigquery.Client(project='samplestudy-465703')

print('準備完了！')

---
## 1. SQL の基本：データを取得してみよう

まずは SQL を使って BigQuery からデータを取得します。

### 1-1. テーブルの中身を見てみる（SELECT / LIMIT）

In [ ]:
# 自転車シェアのトリップ（利用記録）データを10件取得
query = """
SELECT *
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
LIMIT 10
"""

df_sample = client.query(query).to_dataframe()
df_sample

### 1-2. 件数を数える（COUNT）

In [ ]:
# データは全部で何件あるか確認
query = """
SELECT COUNT(*) AS total_trips
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
"""

result = client.query(query).to_dataframe()
print(f"全トリップ数: {result['total_trips'][0]:,} 件")

### 1-3. 条件をつけて絞り込む（WHERE）

In [ ]:
# 利用時間が60分以上のトリップだけを取得（長時間利用）
query = """
SELECT
    trip_id,
    start_station_name,
    end_station_name,
    duration_minutes,
    start_time
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE duration_minutes >= 60
ORDER BY duration_minutes DESC
LIMIT 20
"""

df_long_trips = client.query(query).to_dataframe()
df_long_trips

### 1-4. グループごとに集計する（GROUP BY）

In [ ]:
# 出発駅ごとのトリップ数を集計（上位20駅）
query = """
SELECT
    start_station_name AS station,
    COUNT(*) AS trip_count,
    ROUND(AVG(duration_minutes), 1) AS avg_duration_min
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_station_name IS NOT NULL
GROUP BY start_station_name
ORDER BY trip_count DESC
LIMIT 20
"""

df_stations = client.query(query).to_dataframe()
df_stations

---
## 2. データの可視化：グラフを作ってみよう

取得したデータをグラフにして、視覚的に理解しましょう。

### 2-1. 棒グラフ：人気の駅ランキング

In [ ]:
# 上位15駅を棒グラフで表示
fig, ax = plt.subplots(figsize=(14, 7))

top15 = df_stations.head(15)
bars = ax.barh(range(len(top15)), top15['trip_count'], color=sns.color_palette('viridis', len(top15)))
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['station'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('トリップ数')
ax.set_title('出発駅別トリップ数ランキング（上位15駅）', fontsize=14)

for bar, count in zip(bars, top15['trip_count']):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### 2-2. 折れ線グラフ：月別トリップ数の推移

In [ ]:
# 月別のトリップ数を取得
query = """
SELECT
    FORMAT_TIMESTAMP('%Y-%m', start_time) AS month,
    COUNT(*) AS trip_count
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_time IS NOT NULL
GROUP BY month
ORDER BY month
"""

df_monthly = client.query(query).to_dataframe()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(range(len(df_monthly)), df_monthly['trip_count'], marker='o', markersize=3, linewidth=1.5, color='#2196F3')
ax.fill_between(range(len(df_monthly)), df_monthly['trip_count'], alpha=0.15, color='#2196F3')

# X軸ラベルを間引いて表示
step = max(1, len(df_monthly) // 15)
ax.set_xticks(range(0, len(df_monthly), step))
ax.set_xticklabels(df_monthly['month'].iloc[::step], rotation=45, ha='right', fontsize=9)

ax.set_ylabel('トリップ数')
ax.set_title('月別トリップ数の推移', fontsize=14)
plt.tight_layout()
plt.show()

### 2-3. 円グラフ：利用者タイプの割合

In [ ]:
# 利用者タイプ（会員 vs 一般）の割合
query = """
SELECT
    subscriber_type,
    COUNT(*) AS trip_count
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE subscriber_type IS NOT NULL
    AND subscriber_type != ''
GROUP BY subscriber_type
ORDER BY trip_count DESC
"""

df_subscribers = client.query(query).to_dataframe()

fig, ax = plt.subplots(figsize=(8, 8))
colors = sns.color_palette('Set2', len(df_subscribers))
ax.pie(df_subscribers['trip_count'],
       labels=df_subscribers['subscriber_type'],
       autopct='%1.1f%%',
       colors=colors,
       startangle=90,
       textprops={'fontsize': 11})
ax.set_title('利用者タイプ別の割合', fontsize=14)
plt.show()

### 2-4. ヒートマップ：曜日×時間帯の利用パターン

In [ ]:
# 曜日×時間帯のトリップ数を取得
query = """
SELECT
    EXTRACT(DAYOFWEEK FROM start_time) AS day_of_week,
    EXTRACT(HOUR FROM start_time) AS hour,
    COUNT(*) AS trip_count
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_time IS NOT NULL
GROUP BY day_of_week, hour
ORDER BY day_of_week, hour
"""

df_heatmap = client.query(query).to_dataframe()

# ピボットテーブルに変換
pivot = df_heatmap.pivot(index='day_of_week', columns='hour', values='trip_count').fillna(0)
day_labels = ['日', '月', '火', '水', '木', '金', '土']
pivot.index = day_labels

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(pivot, cmap='YlOrRd', annot=False, fmt='.0f', ax=ax,
            xticklabels=[f'{h}時' for h in range(24)])
ax.set_ylabel('曜日')
ax.set_xlabel('時間帯')
ax.set_title('曜日×時間帯の利用パターン（ヒートマップ）', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. 統計分析：データの特徴を数値で理解しよう

### 3-1. 基本統計量（平均、中央値、標準偏差など）

In [ ]:
# 利用時間のサンプルデータを取得
query = """
SELECT duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE duration_minutes IS NOT NULL
    AND duration_minutes > 0
    AND duration_minutes <= 180
"""

df_duration = client.query(query).to_dataframe()

print('=== 利用時間（分）の基本統計量 ===')
print(f'  データ件数: {len(df_duration):,} 件')
print(f'  平均:       {df_duration["duration_minutes"].mean():.1f} 分')
print(f'  中央値:     {df_duration["duration_minutes"].median():.1f} 分')
print(f'  標準偏差:   {df_duration["duration_minutes"].std():.1f} 分')
print(f'  最小値:     {df_duration["duration_minutes"].min():.1f} 分')
print(f'  最大値:     {df_duration["duration_minutes"].max():.1f} 分')
print()
print('--- pandas の describe() ---')
df_duration.describe()

### 3-2. ヒストグラム：利用時間の分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 全体の分布
axes[0].hist(df_duration['duration_minutes'], bins=60, color='#4CAF50', alpha=0.7, edgecolor='white')
axes[0].axvline(df_duration['duration_minutes'].mean(), color='red', linestyle='--', label=f'平均: {df_duration["duration_minutes"].mean():.1f}分')
axes[0].axvline(df_duration['duration_minutes'].median(), color='blue', linestyle='--', label=f'中央値: {df_duration["duration_minutes"].median():.1f}分')
axes[0].set_xlabel('利用時間（分）')
axes[0].set_ylabel('件数')
axes[0].set_title('利用時間の分布（0〜180分）')
axes[0].legend()

# 30分以下に絞った分布
short = df_duration[df_duration['duration_minutes'] <= 30]
axes[1].hist(short['duration_minutes'], bins=30, color='#FF9800', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('利用時間（分）')
axes[1].set_ylabel('件数')
axes[1].set_title('利用時間の分布（30分以下に拡大）')

plt.tight_layout()
plt.show()

### 3-3. 箱ひげ図：曜日ごとの利用時間の比較

In [ ]:
# 曜日ごとの利用時間（60分以下に絞る）
query = """
SELECT
    EXTRACT(DAYOFWEEK FROM start_time) AS day_of_week,
    duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE duration_minutes IS NOT NULL
    AND duration_minutes > 0
    AND duration_minutes <= 60
    AND start_time IS NOT NULL
"""

df_weekday = client.query(query).to_dataframe()

day_labels = {1: '日', 2: '月', 3: '火', 4: '水', 5: '木', 6: '金', 7: '土'}
df_weekday['day_name'] = df_weekday['day_of_week'].map(day_labels)

fig, ax = plt.subplots(figsize=(10, 6))
day_order = ['月', '火', '水', '木', '金', '土', '日']
sns.boxplot(data=df_weekday, x='day_name', y='duration_minutes', order=day_order, palette='Set3', ax=ax)
ax.set_xlabel('曜日')
ax.set_ylabel('利用時間（分）')
ax.set_title('曜日ごとの利用時間の分布（箱ひげ図）', fontsize=14)
plt.tight_layout()
plt.show()

### 3-4. 相関分析：変数間の関係を調べる

In [ ]:
# 駅ごとの集計データで相関を見る
query = """
SELECT
    start_station_id,
    COUNT(*) AS trip_count,
    AVG(duration_minutes) AS avg_duration,
    COUNT(DISTINCT end_station_id) AS unique_destinations
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_station_id IS NOT NULL
    AND duration_minutes IS NOT NULL
    AND duration_minutes > 0
GROUP BY start_station_id
HAVING COUNT(*) >= 100
"""

df_corr = client.query(query).to_dataframe()

# 相関行列
corr_cols = ['trip_count', 'avg_duration', 'unique_destinations']
corr_matrix = df_corr[corr_cols].corr()

print('=== 相関行列 ===')
print('（1に近いほど正の相関、-1に近いほど負の相関、0に近いほど無相関）')
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 相関行列のヒートマップ
labels = ['トリップ数', '平均利用時間', '行き先の種類数']
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('変数間の相関（ヒートマップ）')

# 散布図
axes[1].scatter(df_corr['trip_count'], df_corr['unique_destinations'],
                alpha=0.6, color='#E91E63', s=40)
axes[1].set_xlabel('トリップ数')
axes[1].set_ylabel('行き先の種類数')
axes[1].set_title('トリップ数 vs 行き先の種類数（散布図）')

plt.tight_layout()
plt.show()

---
## 4. チャレンジ：自分で分析してみよう！

ここまでの内容を参考に、以下のお題に挑戦してみましょう。

### お題1: 一番人気の「到着駅」はどこ？
ヒント: `end_station_name` を `GROUP BY` して `COUNT` する

### お題2: 年ごとのトリップ数はどう変化している？
ヒント: `EXTRACT(YEAR FROM start_time)` で年を取り出す

### お題3: 会員と非会員で利用時間に違いはある？
ヒント: `subscriber_type` で `GROUP BY` して `AVG(duration_minutes)` を比較

下のセルに SQL を書いて試してみてください！

In [ ]:
# ここに自分の SQL を書いてみよう！
query = """
-- ここに SQL を入力
SELECT 'ここを書き換えてね！' AS message
"""

df_challenge = client.query(query).to_dataframe()
df_challenge